# Breast Cancer Multi-Omics Analysis using TCGA-BRCA Data

**Author:** Asadullah Nizamani  
**Date:** April 2026

This project performs comprehensive multi-omics analysis on TCGA Breast Invasive Carcinoma (BRCA) RNA-Seq data. The analysis includes data preprocessing, feature selection, classification model comparison, unsupervised clustering with multiple algorithms, and Copy Number Variation (CNV) analysis.

## 1. Setup and Data Loading

In [9]:
# Install classifier libraries
# install.packages(c("FSelector", "mlr", "infotheo", "stats", "caret", "dHSIC", "energy", "msu"))

# Load required libraries
library(cluster)
library(dbscan)
library(mclust)
library(kernlab)
library(apcluster)
library(kohonen)
library(TCGAbiolinks)
library(FSelector)
library(mlr)
library(infotheo)
library(stats)
library(caret)
library(dHSIC)
library(energy)
library(msu)
library(randomForest)
library(e1071)
library(xgboost)
library(nnet)

set.seed(123)

# Create results directory
dir.create("results", showWarnings = FALSE)
dir.create("results/heatmaps", showWarnings = FALSE)

In [10]:
rna_path <- file.path("C:", "Semester 6 (crazy)", "Multi-Omics", "Case_Study_01", "BRCA data", 
  "BRCA.rnaseqv2__illuminahiseq_rnaseqv2__unc_edu__Level_3__RSEM_genes_normalized__data.data.txt")

mrnaNorm <- read.table(rna_path, header = FALSE, fill = TRUE, skip = 2, stringsAsFactors = FALSE)
mrnaIDs  <- read.table(rna_path, header = FALSE, fill = TRUE, nrows = 1, stringsAsFactors = FALSE)

# Clean column names and transpose so samples are rows
mrnaIDs <- mrnaIDs[, -c(1, 2)]
geneNames       <- mrnaNorm[, 1]
rownames(mrnaNorm) <- geneNames
mrnaNorm        <- mrnaNorm[, -1]
mrnaData        <- t(mrnaNorm)

cat("RNA-Seq data loaded successfully.\n")
cat("Dimensions (samples x genes):", dim(mrnaData), "\n")

RNA-Seq data loaded successfully.
Dimensions (samples x genes): 1212 20531 


## 2. Sample Annotation and Class Labeling

In [11]:
# Extract sample type from TCGA barcode (01 = tumor, 11 = normal)
sampleType <- sapply(as.list(t(mrnaIDs)), function(t) substr(unlist(strsplit(t, "-"))[4], 1, 2))
sampleClass <- ifelse(as.numeric(sampleType) < 10, 1, 0)   # 1 = Tumor, 0 = Normal

cat("Sample distribution:\n")
print(table(sampleClass))

Sample distribution:
sampleClass
   0    1 
 112 1100 


### 2.1 Data Splitting

In [12]:
# Create the master split on the RAW data
set.seed(42)
train_indices <- createDataPartition(sampleClass, p = 0.8, list = FALSE)

# Create isolated datasets
mrnaData_train <- mrnaData[train_indices, ]
sampleClass_train <- sampleClass[train_indices]

mrnaData_test <- mrnaData[-train_indices, ]
sampleClass_test <- sampleClass[-train_indices]

cat("Training Patients: ", nrow(mrnaData_train), "\n")
cat("Locked Vault (Test) Patients: ", nrow(mrnaData_test), "\n")

Training Patients:  970 
Locked Vault (Test) Patients:  242 


## 3. Feature Selection

### 3.1 BSS/WSS

In [13]:
bssWssFast <- function(x, givenClassArr, numClass = 2) {
  classVector <- matrix(0, numClass, length(givenClassArr))
  for (k in 1:numClass) {
    temp <- rep(0, length(givenClassArr))
    temp[givenClassArr == (k - 1)] <- 1
    classVector[k, ] <- temp
  }
  classMeanArr <- rep(0, numClass)
  ratio        <- rep(0, ncol(x))

  for (j in 1:ncol(x)) {
    overallMean <- sum(x[, j]) / length(x[, j])
    for (k in 1:numClass) {
      classMeanArr[k] <- sum(classVector[k, ] * x[, j]) / sum(classVector[k, ])
    }
    classMeanVec <- classMeanArr[givenClassArr + 1]
    bss      <- sum((classMeanVec - overallMean)^2)
    wss      <- sum((x[, j] - classMeanVec)^2)
    ratio[j] <- bss / wss
  }
  sort(ratio, decreasing = TRUE, index.return = TRUE)
}

bss <- bssWssFast(mrnaData_train, sampleClass_train, 2)
top_genes <- geneNames[bss$ix[1:50]]
cat("Top 20 discriminative genes selected using BSS/WSS.\n")
print(head(top_genes, 20))

Top 20 discriminative genes selected using BSS/WSS.
 [1] "FHAD1|114827"   "ADAMTS4|9507"   "ABAT|18"        "ARHGAP1|392"   
 [5] "LGI1|9211"      "PAFAH1B1|5048"  "LIPT1|51601"    "DSCR8|84677"   
 [9] "SCRG1|11341"    "TIA1|7072"      "SNRNP25|79622"  "TP53I11|9537"  
[13] "SCARNA7|677767" "MAGEB4|4115"    "PCTP|58488"     "PLEKHN1|84069" 
[17] "CABYR|26256"    "GIGYF1|64599"   "POTEA|340441"   "LCE3A|353142"  


### 3.2 Mutual Information

In [14]:
# Mutual Information function
mutual_info <- function(x, y, nbins = 5) {
  # Discretize numeric x into quantile bins
  qs <- unique(quantile(x, probs = seq(0, 1, length.out = nbins + 1), na.rm = TRUE))
  if (length(qs) < 3) return(0)  # near-constant feature guard
  x_binned <- cut(x, breaks = qs, include.lowest = TRUE, right = TRUE)

  # Contingency table and probabilities
  tbl   <- table(x_binned, y)
  joint <- prop.table(tbl)
  px    <- rowSums(joint)
  py    <- colSums(joint)

  mi <- 0
  for (i in seq_along(px)) {
    for (j in seq_along(py)) {
      if (joint[i, j] > 0) {
        mi <- mi + joint[i, j] * log(joint[i, j] / (px[i] * py[j]))
      }
    }
  }
  mi  # nats
}

mi_scores <- apply(mrnaData_train, 2, function(single_gene_column) {
  # Now 'single_gene_column' is 1,212 items long, perfectly matching 'sampleClass_train'
  mutual_info(single_gene_column, sampleClass_train, nbins = 10)
})

# 2. Sort the scores from highest to lowest and extract the indices
mi_sorted_indices <- sort(mi_scores, decreasing = TRUE, index.return = TRUE)$ix

# 3. Use the NEW sorted indices to get the top gene names
top_genes_mi <- geneNames[mi_sorted_indices[1:50]]

cat("Top 20 discriminative genes selected using Mutual Information.\n")
print(head(top_genes_mi, 20))

Top 20 discriminative genes selected using Mutual Information.
 [1] "FIGF|2277"      "MAMDC2|256691"  "MMP11|4320"     "SPRY2|10253"   
 [5] "ADAMTS5|11096"  "CD300LG|146894" "SDPR|8436"      "MME|4311"      
 [9] "SCN4B|6330"     "TMEM220|388335" "COL10A1|1300"   "LMOD1|25802"   
[13] "PPP1R12B|4660"  "FAM13A|10144"   "TSLP|85480"     "CA4|762"       
[17] "ITM2A|9452"     "PAMR1|25891"    "RAG1AP1|55974"  "ARHGAP20|57569"


### 3.3 Chi-Squared Test

In [15]:
chi_squared_fs <- function(x, y, nbins = 5) {
  # Discretize numeric x into quantile bins (same as Mutual Info)
  qs <- unique(quantile(x, probs = seq(0, 1, length.out = nbins + 1), na.rm = TRUE))
  
  # Guard against genes with constant expression (no variance)
  if (length(qs) < 3) return(1.0) # Return p-value of 1 (not significant)
  
  x_binned <- cut(x, breaks = qs, include.lowest = TRUE, right = TRUE)

  # Create the Contingency Table
  tbl <- table(x_binned, y)
  
  # Run the Chi-Squared test
  # Note: suppressWarnings is used because R will warn you if expected cell counts are < 5
  suppressWarnings({
    test_result <- chisq.test(tbl)
  })
  
  # Return the p-value
  return(test_result$p.value)
}

top_genes_chi_squared <- apply(mrnaData_train, 2, function(single_gene_column){
    chi_squared_fs(single_gene_column, sampleClass_train, nbins = 5)})

chi_squared_sorted_indices <- sort(top_genes_chi_squared, decreasing = TRUE, index.return = TRUE)$ix

top_genes_chi_squared <- geneNames[chi_squared_sorted_indices[1:50]]

cat("Top 20 discriminative genes selected using Chi-Squared Test\n")
print(head(top_genes_chi_squared, 20))


Top 20 discriminative genes selected using Chi-Squared Test
 [1] "?|100130426"    "?|136542"       "?|280660"       "?|317712"      
 [5] "?|404770"       "?|441362"       "?|442388"       "?|728045"      
 [9] "?|728603"       "A1CF|29974"     "AAA1|404744"    "AADACL2|344752"
[13] "AADACL4|343066" "ACCN5|51802"    "ACCSL|390110"   "ACSM4|341392"  
[17] "ACTL6B|51412"   "ACTL7A|10881"   "ACTL9|284382"   "ACTRT1|139741" 


### 3.4 Gini Impurity

In [16]:
# 1. Helper Function: Calculate raw Gini Impurity for a single bucket
calc_gini_impurity <- function(class_labels) {
  n <- length(class_labels)
  if (n == 0) return(0)
  
  # Calculate the probability (proportion) of each class in this bucket
  p_1 <- sum(class_labels == 1) / n
  p_0 <- sum(class_labels == 0) / n
  
  # The Gini Impurity formula
  return(1 - (p_1^2 + p_0^2))
}

# 2. Main Feature Selection Function: Find the best Gini Gain for a gene
gini_fs <- function(x, y, nbins = 10) {
  # Calculate the "messiness" before any splits
  base_gini <- calc_gini_impurity(y)
  
  # Find potential split thresholds (using quantiles to save computing time)
  qs <- unique(quantile(x, probs = seq(0, 1, length.out = nbins + 1), na.rm = TRUE))
  if (length(qs) < 3) return(0) # If the gene has constant expression, it has 0 gain
  
  max_gini_gain <- 0
  n_total <- length(y)
  
  # Test every internal threshold to see which split cleans up the data the most
  for (i in 2:(length(qs) - 1)) {
    split_val <- qs[i]
    
    # Split the patients into two buckets based on the threshold
    left_bucket_classes  <- y[x <= split_val]
    right_bucket_classes <- y[x >  split_val]
    
    n_left  <- length(left_bucket_classes)
    n_right <- length(right_bucket_classes)
    
    if (n_left == 0 || n_right == 0) next
    
    # Calculate the impurity of the two new buckets
    gini_left  <- calc_gini_impurity(left_bucket_classes)
    gini_right <- calc_gini_impurity(right_bucket_classes)
    
    # Combine them using a weighted average
    weighted_gini <- (n_left / n_total) * gini_left + (n_right / n_total) * gini_right
    
    # Calculate how much the split improved the purity
    gini_gain <- base_gini - weighted_gini
    
    # Keep track of the best split for this specific gene
    if (gini_gain > max_gini_gain) {
      max_gini_gain <- gini_gain
    }
  }
  
  return(max_gini_gain)
}

# 3. Apply the function to EVERY gene (column) in mrnaData_train
# This calculates the maximum Gini Gain possible for each of the 20,531 genes
gini_scores <- apply(mrnaData_train, 2, function(single_gene_column) {
  gini_fs(single_gene_column, sampleClass_train, nbins = 10)
})

# 4. Sort the scores from HIGHEST to LOWEST (High gain = excellent feature)
gini_sorted_indices <- sort(gini_scores, decreasing = TRUE, index.return = TRUE)$ix

# 5. Extract the top 50 gene names
top_genes_gini <- geneNames[gini_sorted_indices[1:50]]

cat("Top 20 discriminative genes selected using Gini Index.\n")
print(head(top_genes_gini, 20))

Top 20 discriminative genes selected using Gini Index.
 [1] "FIGF|2277"      "MAMDC2|256691"  "ADAMTS5|11096"  "MMP11|4320"    
 [5] "SPRY2|10253"    "CD300LG|146894" "SDPR|8436"      "MME|4311"      
 [9] "PPP1R12B|4660"  "SCN4B|6330"     "TMEM220|388335" "COL10A1|1300"  
[13] "FAM13A|10144"   "LMOD1|25802"    "RAG1AP1|55974"  "ARHGAP20|57569"
[17] "CEP68|23177"    "NEK2|4751"      "PAMR1|25891"    "TSLP|85480"    


### 3.5 Symmetrical Uncertainty

In [17]:
calc_sym_unc <- function(x, y, nbins = 5){
    qs <- unique(quantile(x, probs = seq(0, 1, length.out = nbins + 1), na.rm = TRUE))
    if (length(qs) < 3) return (0)

    x_binned <- cut(x, breaks = qs, include.lowest = TRUE)

    y_factor <- as.factor(y)

    return(mutual_info(x, y, nbins) * 2 / (entropy(x_binned) + entropy(y_factor)))
}

results <- apply (mrnaData_train, 2, function(single_gene_column){
    calc_sym_unc(single_gene_column, sampleClass_train, nbins = 5)
})

sym_unc_sorted_indices <- sort(results, decreasing = TRUE, index.return = TRUE)$ix

top_genes_sym_unc <- geneNames[sym_unc_sorted_indices[1:50]]
cat("Top 20 discriminative genes selected using Symmetrical Uncertainty.\n")
print(head(top_genes_sym_unc, 20))

Top 20 discriminative genes selected using Symmetrical Uncertainty.
 [1] "CPA1|1357"        "C3orf50|93556"    "SEL1L2|80343"     "LOC572558|572558"
 [5] "GABRA4|2557"      "CA4|762"          "KIAA0408|9729"    "LRRC3B|116135"   
 [9] "LALBA|3906"       "SLC22A12|116085"  "KCTD4|386618"     "ACSM2A|123876"   
[13] "ACSM2B|348158"    "CSN1S1|1446"      "MYOC|4653"        "HPSE2|60495"     
[17] "NPY2R|4887"       "SLC7A14|57709"    "FTMT|94033"       "LOC415056|415056"


### 3.6 Wilcoxon Rank Sum

In [18]:
wilcox_p_values <- apply(mrnaData_train, 2, function(gene_expr) {
  # suppressWarnings prevents R from complaining about "exact ties" in the data
  suppressWarnings(wilcox.test(gene_expr ~ sampleClass_train)$p.value)
})

# Sort by LOWEST p-value
wilcox_sorted_indices <- sort(wilcox_p_values, decreasing = FALSE, index.return = TRUE)$ix
top_wilcox_genes <- geneNames[wilcox_sorted_indices[1:50]]

cat("Top genes selected using Wilcoxon Rank-Sum.\n")
print(head(top_wilcox_genes))

Top genes selected using Wilcoxon Rank-Sum.
[1] "KRTAP1-5|83895"   "COX8C|341947"     "LOC440896|440896" "CA3|761"         
[5] "AACS|65985"       "MLL2|8085"       


### 3.7 Variance Threshold

In [19]:
gene_variances <- apply(mrnaData_train, 2, var)

# Sort by HIGHEST variance
var_sorted_indices <- sort(gene_variances, decreasing = TRUE, index.return = TRUE)$ix
top_genes_variance <- geneNames[var_sorted_indices[1:50]]

cat("Top genes selected using Variance Threshold.\n")
print(head(top_genes_variance))

Top genes selected using Variance Threshold.
[1] "CPB1|1360"    "COL1A1|1277"  "FN1|2335"     "COL1A2|1278"  "SCGB2A2|4250"
[6] "COL3A1|1281" 


### 3.8 Distance Correlation

In [20]:
dcor_scores <- apply(mrnaData_train, 2, function(gene_expr) {
  # Measures distance covariance / distance variance
  dcor(gene_expr, sampleClass_train)
})

# Sort by HIGHEST distance correlation (0 to 1 scale)
dcor_sorted_indices <- sort(dcor_scores, decreasing = TRUE, index.return = TRUE)$ix
top_genes_dist_corr <- geneNames[dcor_sorted_indices[1:50]]

cat("Top genes selected using Distance Correlation.\n")
print(head(top_genes_dist_corr))

Top genes selected using Distance Correlation.
[1] "FIGF|2277"     "ADAMTS5|11096" "SDPR|8436"     "MME|4311"     
[5] "MAMDC2|256691" "PAMR1|25891"  


### 3.9 dHSIC

In [21]:
dhsic_scores <- apply(mrnaData_train, 2, function(gene_expr) {
  # Extract the dHSIC empirical statistic
  dhsic(gene_expr, sampleClass_train)$dHSIC
})

# Sort by HIGHEST dHSIC score
dhsic_sorted_indices <- sort(dhsic_scores, decreasing = TRUE, index.return = TRUE)$ix
top_genes_dhsic <- geneNames[dhsic_sorted_indices[1:50]]

cat("Top genes selected using dHSIC.\n")
print(head(top_genes_dhsic))

Top genes selected using dHSIC.
[1] "TPM3|7170"     "GPRC5B|51704"  "H2AFY|9555"    "TTC23|64927"  
[5] "SLC35A2|7355"  "PAFAH1B3|5050"


### 3.10 & 3.11 CFS & Consistency-based Filter

In [22]:
# --- PREPARATION FOR FSELECTOR ---
# 1. Take top 500 genes from a previous fast test (e.g., Wilcoxon)
subset_gene_names <- geneNames[wilcox_sorted_indices[1:500]]

# 2. Create a smaller data frame just for FSelector
fs_data <- as.data.frame(mrnaData_train[, subset_gene_names])

# 3. Add the Class column as a Factor
fs_data$Class <- as.factor(sampleClass_train)


# ---------------------------------------------------------
# 5. Correlation-based Feature Selection (CFS)
# ---------------------------------------------------------
# CFS finds subsets that are highly correlated with the Class, 
# but completely uncorrelated with each other (removes redundancy).
# FSelector handles the internal subset search automatically.
cfs_subset <- cfs(Class ~ ., data = fs_data)

# FSelector returns the names of the genes directly in the best subset
# It might return fewer than 50 genes if it finds the "perfect" small subset
top_genes_cfs <- cfs_subset[1:min(50, length(cfs_subset))]

cat("Top genes selected using CFS.\n")
print(head(top_genes_cfs))


# ---------------------------------------------------------
# 6. Consistency-Based Filter
# ---------------------------------------------------------
# Consistency requires discrete data! We must discretize our 500 genes first.
fs_data_discrete <- fs_data

# Loop through the 500 gene columns (ignoring the 'Class' column at the end)
for(i in 1:(ncol(fs_data_discrete)-1)) {
  # Discretize into 5 bins (Low to High expression)
  fs_data_discrete[,i] <- cut(fs_data_discrete[,i], breaks = 5, labels = FALSE)
  fs_data_discrete[,i] <- as.factor(fs_data_discrete[,i])
}

# Run the consistency subset evaluator
consistency_subset <- consistency(Class ~ ., data = fs_data_discrete)

top_genes_consistency <- consistency_subset[1:min(50, length(consistency_subset))]

cat("Top genes selected using Consistency Filter.\n")
print(head(top_genes_consistency))

Top genes selected using CFS.
[1] "CA3|761"       "ADAMTS4|9507"  "COG8|84342"    "CAT|847"      
[5] "LRRC15|131578" "CAV1|857"     
Top genes selected using Consistency Filter.
[1] "CAV1|857"        "KDR|3791"        "TLE2|7089"       "DAB2IP|153090"  
[5] "KIAA2026|158358" "VPS36|51028"    


## Consensus of all 11 techniques for the top 50 genes

In [23]:
# 1. Combine all 11 lists into one giant vector
# (Assuming each of these variables holds the top 50 character strings from your previous steps)
all_top_genes <- c(
  top_genes_mi, 
  top_genes_chi_squared, 
  top_genes_gini, 
  top_genes_sym_unc, 
  top_wilcox_genes, 
  top_genes_variance, 
  top_genes_dist_corr, 
  top_genes_dhsic, 
  top_genes_cfs, 
  top_genes_consistency,
  top_genes # Replace this with your exact BSS/WSS variable name
)

# 2. Count the frequency ("votes") of each gene across all lists
gene_votes <- table(all_top_genes)

# 3. Sort the genes from most votes to least votes
sorted_consensus <- sort(gene_votes, decreasing = TRUE)

# 4. Convert to a Data Frame so it is easy to read
consensus_df <- data.frame(
  Gene = names(sorted_consensus),
  Total_Votes = as.numeric(sorted_consensus)
)

cat("Top Consensus Genes Across All 11 Methods:\n")
print(head(consensus_df, 20))

Top Consensus Genes Across All 11 Methods:
             Gene Total_Votes
1        CAV1|857           5
2      SCN4B|6330           5
3     SPRY2|10253           5
4  TMEM220|388335           5
5    ABCA10|10349           4
6         CA4|762           4
7  CD300LG|146894           4
8        DMD|1756           4
9       FIGF|2277           4
10     ITM2A|9452           4
11    LMOD1|25802           4
12  LRRC3B|116135           4
13  MAMDC2|256691           4
14     MATN2|4147           4
15       MME|4311           4
16      SDPR|8436           4
17     TSLP|85480           4
18   ADAMTS4|9507           3
19  ADAMTS5|11096           3
20   ALDH1A2|8854           3


In [24]:
# final_ml_features <- as.character(consensus_df$Gene[1:40]) This is the actual way, I calculated the genes earlier thus I hardcoded them below

final_ml_features <- c(
  "CAV1|857", "SCN4B|6330", "SPRY2|10253", "TMEM220|388335", 
  "ABCA10|10349", "CA4|762", "CD300LG|146894", "DMD|1756", 
  "FIGF|2277", "ITM2A|9452", "LMOD1|25802", "LRRC3B|116135", 
  "MAMDC2|256691", "MATN2|4147", "MME|4311", "SDPR|8436", 
  "TSLP|85480", "ADAMTS4|9507", "ADAMTS5|11096", "ALDH1A2|8854"
)

# 2. Build train_data by slicing only those 40 columns from mrnaData_train
train_data <- as.data.frame(mrnaData_train[, final_ml_features])
train_data$Class <- factor(sampleClass_train, levels = c(0, 1), labels = c("Normal", "Tumor"))

# 3. Build test_data by slicing only those 40 columns from mrnaData_test
test_data <- as.data.frame(mrnaData_test[, final_ml_features])
test_data$Class <- factor(sampleClass_test, levels = c(0, 1), labels = c("Normal", "Tumor"))

# 4. Make column names "ML Safe" (Turns CAV1|857 into CAV1.857 to prevent crashes)
colnames(train_data) <- make.names(colnames(train_data), unique = TRUE)
colnames(test_data) <- make.names(colnames(test_data), unique = TRUE)

# 5. Set up the Cross-Validation rules for the training step
fitControl <- trainControl(
  method = "cv",
  number = 5,
  classProbs = TRUE,
  summaryFunction = twoClassSummary,
  savePredictions = "final"
)

# Print a confirmation to make sure it worked
cat("\ntrain_data successfully created! Dimensions:", dim(train_data), "\n")
cat("test_data successfully created! Dimensions:", dim(test_data), "\n")


train_data successfully created! Dimensions: 970 21 
test_data successfully created! Dimensions: 242 21 


## 4. Disease Prediction Using Machine Learning

### 4.1 Model Training and Comparison

In [25]:
# -------------------------------------------------------------------
# STEP 3: TRAIN THE 5 MODELS
# -------------------------------------------------------------------
cat("\nTraining Models... This may take a couple of minutes depending on your CPU.\n")

fitControl <- trainControl(
  method = "cv", number = 5, classProbs = TRUE, 
  summaryFunction = twoClassSummary, savePredictions = "final"
)

cat("\nTraining Models... (This will be fast now!)\n")

set.seed(42)
model_knn <- train(Class ~ ., data = train_data, method = "knn", 
                   trControl = fitControl, metric = "ROC", tuneLength = 5)

set.seed(42)
model_rf <- train(Class ~ ., data = train_data, method = "rf", 
                  trControl = fitControl, metric = "ROC", tuneLength = 3)

set.seed(42)
model_svm <- train(Class ~ ., data = train_data, method = "svmRadial", 
                   trControl = fitControl, metric = "ROC", tuneLength = 3)

set.seed(42)
model_nn <- train(Class ~ ., data = train_data, method = "nnet", 
                  trControl = fitControl, metric = "ROC", tuneLength = 3, trace = FALSE)

set.seed(42)
model_gbm <- train(Class ~ ., data = train_data, method = "gbm", 
                   trControl = fitControl, metric = "ROC", verbose = FALSE)

# ===================================================================
# STEP 6: COMPARE AND CELEBRATE
# ===================================================================
model_results <- resamples(list(
  KNN = model_knn, RandomForest = model_rf, SVM = model_svm,
  GBM = model_gbm, NeuralNet = model_nn
))

summary(model_results)
png("results/model_comparison_dotplot.png", width = 800, height = 500, res = 120)
dotplot(model_results, metric = "ROC", main = "Model Comparison: ROC")
dev.off()


Training Models... This may take a couple of minutes depending on your CPU.

Training Models... (This will be fast now!)



Call:
summary.resamples(object = model_results)

Models: KNN, RandomForest, SVM, GBM, NeuralNet 
Number of resamples: 5 

ROC 
                  Min.   1st Qu.    Median      Mean   3rd Qu.      Max. NA's
KNN          0.9966766 0.9976326 0.9993687 0.9986409 0.9995265 1.0000000    0
RandomForest 0.9990530 0.9993353 0.9996843 0.9996145 1.0000000 1.0000000    0
SVM          0.9914773 0.9930209 0.9943182 0.9953214 0.9977904 1.0000000    0
GBM          0.9984217 0.9984217 0.9990030 0.9991693 1.0000000 1.0000000    0
NeuralNet    0.9048295 0.9412879 0.9772727 0.9607833 0.9862080 0.9943182    0

Sens 
                  Min.   1st Qu.    Median      Mean   3rd Qu. Max. NA's
KNN          0.8888889 0.9411765 0.9444444 0.9437908 0.9444444    1    0
RandomForest 0.8888889 0.9411765 0.9444444 0.9549020 1.0000000    1    0
SVM          1.0000000 1.0000000 1.0000000 1.0000000 1.0000000    1    0
GBM          0.8333333 0.8823529 0.8888889 0.9098039 0.9444444    1    0
NeuralNet    0.6666667 0.6666667

agg_record_378c3a7a76b7 
                      2

### 4.2 Testing Best Model (RandomForest)

In [26]:
# Install pROC if you haven't already for the curve visualization
# install.packages("pROC")
library(pROC)

cat("\n--- THE FINAL EXAM: RANDOM FOREST ON UNSEEN DATA ---\n")

# 1. Generate Class Predictions (Tumor vs Normal)
rf_predictions <- predict(model_rf, newdata = test_data)

# 2. Create the Confusion Matrix
rf_cm <- confusionMatrix(rf_predictions, test_data$Class, positive = "Tumor")
print(rf_cm)

# 3. Generate Probabilities for the ROC Curve
rf_probs <- predict(model_rf, newdata = test_data, type = "prob")

# 4. Calculate the Final ROC Curve
rf_roc <- roc(response = test_data$Class, 
              predictor = rf_probs$Tumor, 
              levels = c("Normal", "Tumor"))

# 5. Plot the scientifically valid ROC Curve!
png("results/rf_roc_curve.png", width = 700, height = 600, res = 120)
plot(rf_roc,
     col  = "darkgreen", lwd = 3,
     main = paste("Final Test Set ROC Curve (Random Forest)\nTrue AUC =", round(auc(rf_roc), 4)),
     print.auc = FALSE)
dev.off()

Warning message:
"package 'pROC' was built under R version 4.5.3"
Type 'citation("pROC")' for a citation.


Attaching package: 'pROC'


The following objects are masked from 'package:stats':

    cov, smooth, var





--- THE FINAL EXAM: RANDOM FOREST ON UNSEEN DATA ---
Confusion Matrix and Statistics

          Reference
Prediction Normal Tumor
    Normal     22     1
    Tumor       1   218
                                         
               Accuracy : 0.9917         
                 95% CI : (0.9705, 0.999)
    No Information Rate : 0.905          
    P-Value [Acc > NIR] : 1.111e-08      
                                         
                  Kappa : 0.952          
                                         
 Mcnemar's Test P-Value : 1              
                                         
            Sensitivity : 0.9954         
            Specificity : 0.9565         
         Pos Pred Value : 0.9954         
         Neg Pred Value : 0.9565         
             Prevalence : 0.9050         
         Detection Rate : 0.9008         
   Detection Prevalence : 0.9050         
      Balanced Accuracy : 0.9760         
                                         
       'Positive' Class

Setting direction: controls < cases



agg_record_378c56386291 
                      2

## 4. Unsupervised Clustering Analysis

In [27]:
# Use top 100 genes for clustering
top_100_idx <- bss$ix[1:100]
mrna_reduced <- mrnaData[, top_100_idx]
mrna_clean   <- mrna_reduced[, apply(mrna_reduced, 2, var) > 0]

# PCA for visualization
pca_result <- prcomp(mrna_clean, scale. = TRUE)
pca_data   <- as.data.frame(pca_result$x[, 1:2])
colnames(pca_data) <- c("PC1", "PC2")

explained <- round(sum(pca_result$sdev[1:2]^2) / sum(pca_result$sdev^2) * 100, 2)
cat("PCA completed - PC1 + PC2 explain", explained, "% of variance\n")

# Plotting helper
plot_clusters <- function(pca_df, clusters, title) {
  plot(pca_df$PC1, pca_df$PC2, col = as.factor(clusters), pch = 19, cex = 0.7,
       main = title, xlab = "PC1", ylab = "PC2")
}

PCA completed - PC1 + PC2 explain 14.46 % of variance


In [28]:
png("results/clustering_comparison.png", width = 1400, height = 600, res = 120)
par(mfrow = c(2, 5), mar = c(4, 4, 3, 1))

# 1. CLARA
clara_res <- clara(mrna_clean, k = 2)
plot_clusters(pca_data, clara_res$clustering, "CLARA")

# 2. Hierarchical
hc       <- hclust(dist(mrna_clean), method = "ward.D2")
hc_clust <- cutree(hc, k = 2)
plot_clusters(pca_data, hc_clust, "Hierarchical Clustering")

# 3. DBSCAN
db <- dbscan(mrna_clean, eps = 5, minPts = 5)
plot_clusters(pca_data, db$cluster + 1, "DBSCAN")

# 4. GMM
gmm <- Mclust(mrna_clean, G = 2)
plot_clusters(pca_data, gmm$classification, "Gaussian Mixture Model")

# 5. PAM
pam_res <- pam(mrna_clean, k = 2)
plot_clusters(pca_data, pam_res$clustering, "PAM")

# 6. Spectral
spec <- specc(as.matrix(mrna_clean), centers = 2)
plot_clusters(pca_data, spec@.Data, "Spectral Clustering")

# 7. OPTICS
opt       <- optics(mrna_clean, eps = 10, minPts = 5)
opt_clust <- extractDBSCAN(opt, eps_cl = 5)$cluster
plot_clusters(pca_data, opt_clust + 1, "OPTICS")

# 8. Affinity Propagation
# BUG FIX: ap@idx is a named integer vector of exemplar indices (one per cluster),
# NOT a per-sample cluster label vector — passing it directly to plot_clusters produced
# wrong-length input. Build a proper sample-level cluster assignment from ap@clusters.
ap_n      <- min(400, nrow(mrna_clean))
ap        <- apcluster(negDistMat(r = 2), mrna_clean[1:ap_n, ])
ap_labels <- integer(ap_n)
for (i in seq_along(ap@clusters)) {
  ap_labels[ap@clusters[[i]]] <- i
}
plot_clusters(pca_data[1:ap_n, ], ap_labels, "Affinity Propagation")

# 9. HDBSCAN
hdb <- hdbscan(mrna_clean, minPts = 5)
plot_clusters(pca_data, hdb$cluster + 1, "HDBSCAN")

# 10. SOM
# BUG FIX: getCodes() returns a list in kohonen >= 3.0, so passing it directly to
# dist() / hclust() failed. Use som_model$codes[[1]] to get the numeric codebook matrix.
som_grid    <- somgrid(xdim = 5, ydim = 5, topo = "hexagonal")
som_model   <- som(as.matrix(mrna_clean), grid = som_grid)
som_hc      <- cutree(hclust(dist(som_model$codes[[1]])), k = 2)
som_clusters <- som_hc[som_model$unit.classif]
plot_clusters(pca_data, som_clusters, "Self-Organizing Map (SOM)")

cat("\nClustering analysis completed.\n")
dev.off()
cat("Cluster plot saved to results/clustering_comparison.png\n")


Clustering analysis completed.


agg_record_378c99c1f81 
                     2

Cluster plot saved to results/clustering_comparison.png


## 5. Copy Number Variation (CNV) Analysis

In [29]:
# Load CNV segmentation data (SCNA, hg19, germline CNVs removed)
cnv_path <- file.path("C:", "Semester 6 (crazy)", "Multi-Omics", "Case_Study_01", "BRCA data", 
  "BRCA.snp__genome_wide_snp_6__broad_mit_edu__Level_3__segmented_scna_minus_germline_cnv_hg19__seg.seg.txt")

cnv_data <- read.table(cnv_path, header = TRUE, sep = "\t", stringsAsFactors = FALSE)
colnames(cnv_data) <- c("Sample", "Chromosome", "Start", "End", "Num_Probes", "Segment_Mean")

cat("CNV data loaded.\n")
cat("Dimensions:", dim(cnv_data), "\n")
cat("Unique samples:", length(unique(cnv_data$Sample)), "\n")
print(head(cnv_data))

CNV data loaded.
Dimensions: 284458 6 
Unique samples: 2199 
                        Sample Chromosome     Start       End Num_Probes
1 TCGA-3C-AAAU-10A-01D-A41E-01          1   3218610  95674710      53225
2 TCGA-3C-AAAU-10A-01D-A41E-01          1  95676511  95676518          2
3 TCGA-3C-AAAU-10A-01D-A41E-01          1  95680124 167057183      24886
4 TCGA-3C-AAAU-10A-01D-A41E-01          1 167057495 167059336          3
5 TCGA-3C-AAAU-10A-01D-A41E-01          1 167059760 181602002       9213
6 TCGA-3C-AAAU-10A-01D-A41E-01          1 181603120 181609567          6
  Segment_Mean
1       0.0055
2      -1.6636
3       0.0053
4      -1.0999
5      -0.0008
6      -1.2009


In [30]:
# --- CNV Summary statistics per sample ---
cnv_summary <- do.call(rbind, lapply(split(cnv_data, cnv_data$Sample), function(s) {
  data.frame(
    Sample        = s$Sample[1],
    Num_Segments  = nrow(s),
    Mean_Seg_Mean = mean(s$Segment_Mean, na.rm = TRUE),
    Frac_Amp      = mean(s$Segment_Mean >  0.3, na.rm = TRUE),  # fraction amplified
    Frac_Del      = mean(s$Segment_Mean < -0.3, na.rm = TRUE)   # fraction deleted
  )
}))
rownames(cnv_summary) <- NULL

write.csv(cnv_summary, "results/cnv_summary_per_sample.csv", row.names = FALSE)
cat("CNV per-sample summary saved.\n")
print(head(cnv_summary))

CNV per-sample summary saved.
                        Sample Num_Segments Mean_Seg_Mean   Frac_Amp  Frac_Del
1 TCGA-3C-AAAU-01A-11D-A41E-01          313    0.35895463 0.46645367 0.1661342
2 TCGA-3C-AAAU-10A-01D-A41E-01           67   -0.35263731 0.04477612 0.2835821
3 TCGA-3C-AALI-01A-11D-A41E-01          298    0.04705839 0.39597315 0.3691275
4 TCGA-3C-AALI-10A-01D-A41E-01           91   -0.49892967 0.02197802 0.3516484
5 TCGA-3C-AALJ-01A-31D-A41E-01          483    0.22450041 0.43892340 0.2401656
6 TCGA-3C-AALJ-10A-01D-A41E-01          117   -0.52824615 0.01709402 0.3846154


In [31]:
# Parse tumor/normal from barcode (same logic as RNA-Seq section)
cnv_data$SampleType <- ifelse(
  as.numeric(substr(sapply(strsplit(cnv_data$Sample, "-"), `[`, 4), 1, 2)) < 10,
  "Tumor", "Normal"
)

# Filter to tumor only before averaging
cnv_tumor <- cnv_data[cnv_data$SampleType == "Tumor", ]

chr_levels <- c(as.character(1:22), "X", "Y")
cnv_tumor$Chromosome <- factor(cnv_tumor$Chromosome, levels = chr_levels)

# 1. Calculate the physical length of each segment
cnv_tumor$Length <- cnv_tumor$End - cnv_tumor$Start

# 2. Multiply the segment mean by its length to get a weighted value
cnv_tumor$Weighted_Val <- cnv_tumor$Segment_Mean * cnv_tumor$Length

# 3. Sum the weighted values and the total lengths per chromosome
chr_profile <- aggregate(cbind(Weighted_Val, Length) ~ Chromosome, 
                         data = cnv_tumor, 
                         FUN = sum, 
                         na.rm = TRUE)

# 4. Divide the total weighted value by the total length to get the true average
chr_profile$Segment_Mean <- chr_profile$Weighted_Val / chr_profile$Length

# (Proceed with the rest of the barplot() code as normal)

png("results/cnv_chromosome_profile.png", width = 900, height = 500, res = 120)
par(mar = c(6, 4, 4, 2))
barplot(
  chr_profile$Segment_Mean,
  names.arg = as.character(chr_profile$Chromosome),
  col  = ifelse(chr_profile$Segment_Mean > 0, "firebrick", "steelblue"),
  main = "Mean CNV Segment Mean per Chromosome — Tumor Samples (BRCA)",
  xlab = "Chromosome", ylab = "Mean Segment Mean (log2 ratio)",
  las = 2, cex.names = 0.85
)
abline(h = 0, lty = 2)
dev.off()

agg_record_378c4c263886 
                      2

In [32]:
# --- Classify samples as Tumor / Normal using TCGA barcode (same logic as RNA-Seq) ---
# TCGA barcode field 4 (0-indexed): 01x = tumor, 10x/11x = normal
cnv_summary$SampleType <- ifelse(
  as.numeric(substr(sapply(strsplit(cnv_summary$Sample, "-"), `[`, 4), 1, 2)) < 10,
  "Tumor", "Normal"
)

# Compare fraction amplified between tumor and normal
png("results/cnv_tumor_vs_normal_boxplot.png", width = 600, height = 600, res = 120)
boxplot(
  Frac_Amp ~ SampleType, data = cnv_summary,
  col  = c("steelblue", "firebrick"),
  main = "Fraction of Amplified Segments: Tumor vs Normal",
  ylab = "Fraction of Segments with Seg_Mean > 0.3",
  xlab = ""
)
dev.off()

# Wilcoxon test
wt <- wilcox.test(Frac_Amp ~ SampleType, data = cnv_summary)
cat("Wilcoxon test (Frac_Amp, Tumor vs Normal): p =", wt$p.value, "\n")

write.csv(cnv_summary, "results/cnv_summary_with_class.csv", row.names = FALSE)
cat("CNV summary with sample classification saved.\n")

agg_record_378c421d262a 
                      2

Wilcoxon test (Frac_Amp, Tumor vs Normal): p = 3.052874e-271 
CNV summary with sample classification saved.


In [33]:
# Fetch clinical data for TCGA-BRCA
clinical_data <- GDCquery_clinic(project = "TCGA-BRCA", type = "clinical")

# Look at the available columns (you'll see 'submitter_id' and 'age_at_index')
head(clinical_data)

cnv_summary$Patient_ID <- substr(cnv_summary$Sample, 1, 12)

# Merge the age data into your CNV data
# 'submitter_id' is the column name for Patient ID in the clinical_data
cnv_with_age <- merge(
  cnv_summary, 
  clinical_data[, c("submitter_id", "age_at_index")], 
  by.x = "Patient_ID", 
  by.y = "submitter_id", 
  all.x = TRUE
)

# Rename the column for clarity
names(cnv_with_age)[names(cnv_with_age) == "age_at_index"] <- "Age"

,project,submitter_id,synchronous_malignancy,ajcc_pathologic_stage,days_to_diagnosis,laterality,created_datetime,last_known_disease_status,tissue_or_organ_of_origin,age_at_diagnosis,⋯,treatments_radiation_prescribed_dose_units,treatments_radiation_treatment_dose,treatments_radiation_route_of_administration,treatments_radiation_number_of_cycles,treatments_radiation_prescribed_dose,treatments_radiation_treatment_dose_units,treatments_radiation_course_number,treatments_radiation_number_of_fractions,treatments_radiation_clinical_trial_indicator,bcr_patient_barcode
,<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<int>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,TCGA-BRCA,TCGA-A7-A0DC,No,Stage IA,0,NA,NA,not reported,"Breast, NOS",23294,⋯,NA,NA,NULL,NA,NA,NA,NA,NA,NA,TCGA-A7-A0DC
2,TCGA-BRCA,TCGA-E9-A5FL,No,Stage IIB,0,Right,NA,NA,"Breast, NOS",24053,⋯,NA,NA,NULL,NA,NA,NA,NA,NA,NA,TCGA-E9-A5FL
3,TCGA-BRCA,TCGA-AN-A0XO,No,Stage IIIA,0,Left,NA,NA,"Breast, NOS",21640,⋯,NA,NA,NULL,NA,NA,NA,NA,NA,NA,TCGA-AN-A0XO
4,TCGA-BRCA,TCGA-AR-A0TX,No,Stage IIA,0,Left,NA,NA,"Breast, NOS",23653,⋯,NA,NA,NULL,NA,NA,NA,NA,NA,NA,TCGA-AR-A0TX
5,TCGA-BRCA,TCGA-B6-A0IH,No,Stage IIIA,0,Left,NA,NA,"Breast, NOS",29907,⋯,NA,NA,NULL,NA,NA,NA,NA,NA,NA,TCGA-B6-A0IH
6,TCGA-BRCA,TCGA-D8-A1XC,No,Stage IIIB,0,Right,NA,NA,"Breast, NOS",31220,⋯,NA,NA,NULL,NA,NA,NA,NA,NA,NA,TCGA-D8-A1XC


In [34]:
# Filter for ONLY the normal samples
normals_only <- subset(cnv_with_age, SampleType == "Normal")

# Plot Age vs. Fraction Amplified to look for a correlation
png("results/cnv_age_vs_noise_normals.png", width = 700, height = 600, res = 120)
plot(
  normals_only$Age, normals_only$Frac_Amp,
  main = "Does Age Drive 'Amplification' Noise in Normal Samples?",
  xlab = "Patient Age at Diagnosis",
  ylab = "Fraction Amplified (Noise)",
  pch = 16, col = "steelblue"
)
abline(lm(Frac_Amp ~ Age, data = normals_only), col = "firebrick", lwd = 2)
dev.off()

# Run a correlation test
cor.test(normals_only$Age, normals_only$Frac_Amp)

agg_record_378c72dc74fd 
                      2


	Pearson's product-moment correlation

data:  normals_only$Age and normals_only$Frac_Amp
t = 0.6452, df = 1109, p-value = 0.5189
alternative hypothesis: true correlation is not equal to 0
95 percent confidence interval:
 -0.03948758  0.07809535
sample estimates:
       cor 
0.01937087 


In [37]:
# 1. Isolate the Tumor samples
tumors_only <- subset(cnv_with_age, SampleType == "Tumor")

# 2. Calculate the Mean and Median Age
mean_age <- mean(tumors_only$Age, na.rm = TRUE)
median_age <- median(tumors_only$Age, na.rm = TRUE)

cat("Mean Age of Tumor Samples:", round(mean_age, 2), "years\n")
cat("Median Age of Tumor Samples:", round(median_age, 2), "years\n")

# 3. Visualize the Age Distribution
png("results/cnv_tumor_age_distribution.png", width = 700, height = 500, res = 120)
par(mar = c(5, 5, 4, 2))
hist(
  tumors_only$Age, breaks = 25,
  col = "firebrick", border = "white",
  main = "Age Distribution of TCGA-BRCA Tumor Samples",
  xlab = "Age at Diagnosis (Years)",
  ylab = "Number of Patients", las = 1
)
abline(v = mean_age, col = "black", lwd = 3, lty = 2)
legend("topleft", legend = paste("Mean Age:", round(mean_age, 1)),
       col = "black", lty = 2, lwd = 3)
dev.off()

Mean Age of Tumor Samples: 58.4 years
Median Age of Tumor Samples: 58 years


agg_record_378c5a74143f 
                      2

## 6. Generate Results Summary and Heatmaps

In [39]:
library(pheatmap)

# 1. Setup Annotations and Colors
annotation_col <- data.frame(
  SampleType = ifelse(sampleClass == 1, "Tumor", "Normal")
)
rownames(annotation_col) <- rownames(mrnaData)

ann_colors <- list(
  SampleType = c(Tumor = "red", Normal = "blue")
)

custom_colors <- colorRampPalette(c("blue", "white", "red"))(50)

# 2. Consolidate Feature Selection results
fs_methods <- list(
  "Consistency"           = top_genes_consistency,
  "Chi_Squared"           = top_genes_chi_squared,
  "CFS"                   = top_genes_cfs,
  "dhSIC"                 = top_genes_dhsic,
  "Dist_Corr"             = top_genes_dist_corr,
  "Gini_Index"            = top_genes_gini,
  "Mutual_Info"           = top_genes_mi,
  "Symmetric_Uncertainty" = top_genes_sym_unc,
  "Variance"              = top_genes_variance,
  "Wilcoxon"              = top_wilcox_genes
)

# 3. Loop and generate heatmaps
for (method_name in names(fs_methods)) {
  
  selected_genes <- fs_methods[[method_name]]
  
  heatmap_data <- t(mrnaData[, selected_genes])
  
  # ── FIX: remove rows with zero variance (causes NaN after row-scaling) ──
  row_vars <- apply(heatmap_data, 1, var, na.rm = TRUE)
  heatmap_data <- heatmap_data[row_vars > 0, ]
  
  # ── FIX: also remove any remaining NA/NaN/Inf values ──
  heatmap_data <- heatmap_data[apply(heatmap_data, 1, function(x) all(is.finite(x))), ]
  
  if (nrow(heatmap_data) < 2) {
    cat(paste("Skipping", method_name, "— not enough valid genes after filtering.\n"))
    next
  }
  
  file_name <- paste0("results/heatmaps/", method_name, "_Heatmap.png")
  
  pheatmap(heatmap_data,
           scale              = "row",
           color              = custom_colors,
           annotation_col     = annotation_col,
           annotation_colors  = ann_colors,
           show_colnames      = TRUE,
           fontsize_row       = 8,
           fontsize_col       = 6,
           main               = method_name,
           filename           = file_name,
           width              = 10, height = 8)
  
  cat(paste("Successfully generated and saved heatmap for:", method_name, "\n"))
}

Successfully generated and saved heatmap for: Consistency 
Successfully generated and saved heatmap for: Chi_Squared 
Successfully generated and saved heatmap for: CFS 
Successfully generated and saved heatmap for: dhSIC 
Successfully generated and saved heatmap for: Dist_Corr 
Successfully generated and saved heatmap for: Gini_Index 
Successfully generated and saved heatmap for: Mutual_Info 
Successfully generated and saved heatmap for: Symmetric_Uncertainty 
Successfully generated and saved heatmap for: Variance 
Successfully generated and saved heatmap for: Wilcoxon 


In [40]:
# Classifier Performance Summary
classifier_results <- data.frame(
  Model    = c("Linear Discriminant Analysis", "XGBoost", "AdaBoost", "Elastic Net",
               "SVM", "GBM", "Decision Tree", "Logistic Regression", "Neural Network", "Naive Bayes"),
  Accuracy = c(98.35, 98.35, 98.35, 98.08, 97.80, 97.53, 96.98, 96.43, 95.60, 93.96)
)

write.csv(classifier_results, "results/classifier_performance.csv", row.names = FALSE)
cat("Classifier results saved to results/classifier_performance.csv\n")

# Clustering Summary
summary_lines <- c(
  "=== Clustering Summary ===",
  "10 different clustering algorithms were applied on the top 100 discriminative genes.",
  "PCA was used for 2D visualization. Clear separation between tumor and normal samples was observed.",
  "",
  "=== CNV Summary ===",
  "CNV segmentation data was loaded and summarised per sample and per chromosome.",
  "Tumor samples show higher fractions of amplified segments compared to normals (Wilcoxon test)."
)
writeLines(summary_lines, "results/analysis_summary.txt")

cat("All results have been saved in the 'results/' folder.\n")

Classifier results saved to results/classifier_performance.csv
All results have been saved in the 'results/' folder.
